# DATA09 多标签单事件特征提取

本 notebook 用于对 `E:\codes\ZZ-BK\DATA09` 下多个标签目录中的 npz 单事件样本进行特征提取。

## 1. 概览

- 输入：带标签的 npz 事件文件，通常包含两个 `phase_data` 通道。
- 时间原点：读取每个文件头信息中的 `arrival_time`。
- 每个源文件截取三个窗口：`[-10 ms, 20 ms]`、`[0 ms, 30 ms]`、`[5 ms, 35 ms]`。
- 输出行：一个截取窗口对应一行特征，并写入 `label` 与 `window_mode`。
- 输出分片：每个标签至少输出一个 CSV；同一标签每处理 `MAX_FILES_PER_OUTPUT` 个源数据文件后新建一个 CSV。
- 增量持久化：每个源数据文件处理完成后，立即把特征行追加写入本地 CSV，并追加一行本地处理日志。


## 2. 环境初始化

In [1]:
from __future__ import annotations

import os
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# 默认允许底层特征模块在可用时使用 GPU；具体是否在单事件窗口中启用共享 STFT 由 ENABLE_SHARED_STFT 控制。
os.environ.setdefault('FEA_CPT_USE_GPU', '1')

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt_gpu_v2_2.sliding_window import gpu_backend_info, _auto_detect_workers
from data09_single_event import (
    get_cut_windows,
    load_npz_channel,
    parse_npz_ts,
    process_event_record_timed,
)

print(f'工作目录: {workspace}')
print(gpu_backend_info())
print(f'Python: {sys.version}')
print(f'CPU 核心数: {os.cpu_count()}')
print(f'推荐并行进程数: {_auto_detect_workers()}')
print('特征提取模块加载完成')


工作目录: e:\codes\ZZ-BK
[GPU] 使用 CUDA: NVIDIA GeForce RTX 4050 Laptop GPU
Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
CPU 核心数: 16
推荐并行进程数: 14
特征提取模块加载完成


## 3. 全局配置

In [2]:
# 多标签数据路径。需要增删标签时，只修改此列表。
DATASETS = [
    {'data_root': Path(r'E:\codes\ZZ-BK\DATA09\v0-bk'), 'label': 'BK00'},
    {'data_root': Path(r'E:\codes\ZZ-BK\DATA09\v0-qj'), 'label': 'QJ00'},
    {'data_root': Path(r'E:\codes\ZZ-BK\DATA09\v05-bk'), 'label': 'BK05'},
    {'data_root': Path(r'E:\codes\ZZ-BK\DATA09\v05-qj'), 'label': 'QJ05'},
]

# 每个标签下每处理多少个源 npz 文件新建一个输出 CSV。
MAX_FILES_PER_OUTPUT = 100

# 文件级 joblib 进程并行数；None 表示自动按 CPU 核心数检测。
FILE_WORKERS = None

# 短事件窗口通常更适合 CPU 多进程并行。
# 只有在本机测试确认 GPU/shared STFT 更快时，才建议改为 True。
ENABLE_SHARED_STFT = False

# 为 True 时，重新运行当前 RUN_TIMESTAMP 的批处理单元会先清理同批次输出，避免重复追加。
OVERWRITE_CURRENT_RUN_OUTPUTS = True

# 指定读取通道；当前 npz 的 phase_data 通常为二维数组，CHANNEL_INDEX=0 表示第一通道。
TARGET_CHANNEL = 'phase_data'
CHANNEL_INDEX = 0

TARGET_SAMPLE_RATE = 1_000_000.0
PREPROC_BAND = (1_000.0, 400_000.0)

# 特征计算频带配置。
BANDS = [
    ('b_1k_100k',  (1_000.0,  100_000.0)),
    ('b_1k_10k',   (1_000.0,  10_000.0)),
    ('b_10k_20k',  (10_000.0, 20_000.0)),
    ('b_20k_30k',  (20_000.0, 30_000.0)),
    ('b_30k_40k',  (30_000.0, 40_000.0)),
    ('b_40k_60k',  (40_000.0, 60_000.0)),
    ('b_10k_50k',  (10_000.0, 50_000.0)),
    ('b_1k_50k',   (1_000.0, 50_000.0)),
]

# 到时窗口配置：以 arrival_time 为 0 ms。
# CUT_MODES = [
#     ('m10_20', -10.0, 20.0),
#     ('0_30',     0.0, 30.0),
#     ('5_30',     5.0, 35.0),
# ]

CUT_MODES = [
    ('m20_10', -20.0, 10.0),
    ('m15_15', -15.0, 15.0),
    ('m10_20', -10.0, 20.0),
    ('m5_20', -5.0, 25.0),
    ('0_30',     0.0, 30.0),
    ('5_30',     5.0, 35.0),
]

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_BASE_ROOT = workspace / 'outputs' / 'DATA09_multi_label_single_event_features'
PROCESS_LOG_DIR = OUTPUT_BASE_ROOT / '_process_logs'
PROCESS_LOG_CSV = PROCESS_LOG_DIR / f'process_log_{RUN_TIMESTAMP}.csv'
OUTPUT_BASE_ROOT.mkdir(parents=True, exist_ok=True)
PROCESS_LOG_DIR.mkdir(parents=True, exist_ok=True)

for cfg in DATASETS:
    cfg['output_root'] = OUTPUT_BASE_ROOT / f'{cfg["data_root"].name}_features_{cfg["label"]}'
    cfg['output_root'].mkdir(parents=True, exist_ok=True)

print('数据集配置:')
for cfg in DATASETS:
    print(f'  {cfg["label"]}: {cfg["data_root"]} -> {cfg["output_root"]}')
print(f'每个输出 CSV 最多包含源文件数: {MAX_FILES_PER_OUTPUT}')
print(f'文件级并行进程数: {_auto_detect_workers() if FILE_WORKERS is None else int(FILE_WORKERS)}')
print(f'单事件窗口启用 shared STFT/GPU: {ENABLE_SHARED_STFT}')
print(f'处理日志: {PROCESS_LOG_CSV}')
print(f'目标采样率: {TARGET_SAMPLE_RATE/1e3:.0f} kHz')
print(f'频带数量: {len(BANDS)}')
print('截取窗口:')
for name, a, b in CUT_MODES:
    print(f'  {name}: [{a:+.0f} ms, {b:+.0f} ms]')


数据集配置:
  BK00: E:\codes\ZZ-BK\DATA09\v0-bk -> e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-bk_features_BK00
  QJ00: E:\codes\ZZ-BK\DATA09\v0-qj -> e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v0-qj_features_QJ00
  BK05: E:\codes\ZZ-BK\DATA09\v05-bk -> e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-bk_features_BK05
  QJ05: E:\codes\ZZ-BK\DATA09\v05-qj -> e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\v05-qj_features_QJ05
每个输出 CSV 最多包含源文件数: 100
文件级并行进程数: 14
单事件窗口启用 shared STFT/GPU: False
处理日志: e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\_process_logs\process_log_20260906_122934.csv
目标采样率: 1000 kHz
频带数量: 8
截取窗口:
  m20_10: [-20 ms, +10 ms]
  m15_15: [-15 ms, +15 ms]
  m10_20: [-10 ms, +20 ms]
  m5_20: [-5 ms, +25 ms]
  0_30: [+0 ms, +30 ms]
  5_30: [+5 ms, +35 ms]


## 4. 源文件发现

In [3]:
source_files_by_label = {}
source_records = []

for cfg in DATASETS:
    data_root = cfg['data_root']
    label = cfg['label']
    files = sorted(data_root.glob('*.npz'))
    if not files:
        raise FileNotFoundError(f'标签 {label} 未找到 .npz 文件: {data_root}')
    source_files_by_label[label] = files
    for file_idx, fp in enumerate(files):
        # output_part 按同一标签内的源数据文件序号计算，而不是按特征行数计算。
        part_idx = file_idx // MAX_FILES_PER_OUTPUT + 1
        source_records.append({
            'label': label,
            'data_root': data_root,
            'path': fp,
            'file_index_in_label': file_idx,
            'output_part': part_idx,
        })

source_files = [record['path'] for record in source_records]

print(f'共发现 {len(source_files)} 个源文件，标签数: {len(source_files_by_label)}')
for label, files in source_files_by_label.items():
    part_count = int(np.ceil(len(files) / MAX_FILES_PER_OUTPUT))
    print(f'\n{label}: {len(files)} 个源文件，{part_count} 个输出分片')
    for f in files:
        print(f'  {f.name}')


共发现 195 个源文件，标签数: 4

BK00: 17 个源文件，1 个输出分片
  BK00-FIP-1000K-20260829T151538.488.npz
  BK00-FIP-1000K-20260829T151922.470.npz
  BK00-FIP-1000K-20260829T152226.360.npz
  BK00-FIP-1000K-20260829T152312.647.npz
  BK00-FIP-1000K-20260829T152356.900.npz
  BK00-FIP-1000K-20260829T152452.671.npz
  BK00-FIP-1000K-20260829T152626.333.npz
  BK00-FIP-1000K-20260829T152700.614.npz
  BK00-FIP-1000K-20260829T152752.781.npz
  BK00-FIP-1000K-20260829T152855.599.npz
  BK00-FIP-1000K-20260904T171346.119.npz
  BK00-FIP-1000K-20260904T171457.821.npz
  BK00-FIP-1000K-20260904T171609.505.npz
  BK00-FIP-1000K-20260904T171647.501.npz
  BK00-FIP-1000K-20260904T171859.439.npz
  BK00-FIP-1000K-20260904T172000.589.npz
  BK00-FIP-1000K-20260904T172108.347.npz

QJ00: 108 个源文件，2 个输出分片
  QJ00-FIP-1000K-20260829T151237.470.npz
  QJ00-FIP-1000K-20260829T151238.108.npz
  QJ00-FIP-1000K-20260829T151238.710.npz
  QJ00-FIP-1000K-20260829T151619.049.npz
  QJ00-FIP-1000K-20260829T151620.050.npz
  QJ00-FIP-1000K-20260829T15162

## 5. 头文件检查

In [4]:
first_record = source_records[0]
test_file = first_record['path']
print(f'检查文件: {first_record["label"]} / {test_file.name}')

with np.load(test_file, allow_pickle=True) as data:
    print(f'\n键: {list(data.keys())}')
    print(f'phase_data 形状: {data["phase_data"].shape}, dtype: {data["phase_data"].dtype}')
    print(f'通道名: {data["channel_names"]}')
    print(f'采样率: {float(np.asarray(data["sample_rate"]).item()):.0f} Hz')
    print(f'点数: {int(np.asarray(data["npts"]).item())}')
    print(f'开始时间 starttime: {str(data["starttime"])}')
    print(f'到时 arrival_time: {str(data["arrival_time"])}')
    print(f'标签 type: {str(data["type"])}')
    print(f'data_info: {data["data_info"]}')


检查文件: BK00 / BK00-FIP-1000K-20260829T151538.488.npz

键: ['phase_data', 'channels', 'channel_names', 'channel_count', 'sample_rate', 'comm_count', 'npts', 'timestamp', 'starttime', 'arrival_time', 'type', 'data_info']
phase_data 形状: (400001, 2), dtype: float64
通道名: ['phase_data' 'phase_data_ch2']
采样率: 1000000 Hz
点数: 400001
开始时间 starttime: 20260829T151538.488
到时 arrival_time: 20260829151538.6942
标签 type: BK00
data_info: {'type': 'phase_data_export_visible_segment', 'length': 400001, 'npts': 400001, 'duration_seconds': 0.400001, 'save_time': '2026-09-03T22:03:48.483', 'starttime': '20260829T151538.488', 'arrival_time': '20260829151538.6942', 'sample_type': 'BK00', 'channel_count': 2, 'channel_names': ('phase_data', 'phase_data_ch2')}


## 6. 单文件测试

In [5]:
first_record = source_records[0]
test_file = first_record['path']
test_label = first_record['label']
print(f'测试文件: {test_label} / {test_file.name}')

src = load_npz_channel(test_file, channel_index=CHANNEL_INDEX)
signal = src['signal']
fs = src['sample_rate']

arrival_offset_s = (parse_npz_ts(src['arrival_time_raw']) - parse_npz_ts(src['starttime_raw'])).total_seconds()
print(f'  采样率: {fs/1e3:.0f} kHz, 样本点数: {len(signal):,}')
print(f'  通道: {src["channel_names"]} -> index {src["channel_index"]}')
print(f'  到时相对采集起点偏移: {arrival_offset_s*1000:.2f} ms')

cut_windows = get_cut_windows(len(signal), fs, arrival_offset_s, CUT_MODES)
print('\n截取窗口:')
for name, i0, i1 in cut_windows:
    print(f'  {name}: [{i0}, {i1}) -> {i1-i0} 点 ({(i1-i0)/fs*1000:.2f} ms)')

print('\n开始计算特征...')
t0 = time.time()
test_rows, test_err, worker_elapsed = process_event_record_timed(
    first_record,
    BANDS,
    CUT_MODES,
    channel_index=CHANNEL_INDEX,
    enable_shared_stft=ENABLE_SHARED_STFT,
)
elapsed = time.time() - t0
if test_err is not None:
    raise RuntimeError(test_err)

df_test = pd.DataFrame(test_rows)
print(f'  总耗时: {elapsed:.1f} s')
print(f'  worker 耗时: {worker_elapsed:.1f} s')
print(f'  特征行数: {len(df_test)}')
print(f'  特征列数: {len(df_test.columns) - 13}')
print('\n单文件测试通过')


测试文件: BK00 / BK00-FIP-1000K-20260829T151538.488.npz
  采样率: 1000 kHz, 样本点数: 400,001
  通道: ['phase_data', 'phase_data_ch2'] -> index 0
  到时相对采集起点偏移: 206.20 ms

截取窗口:
  m20_10: [186200, 216200) -> 30000 点 (30.00 ms)
  m15_15: [191200, 221200) -> 30000 点 (30.00 ms)
  m10_20: [196200, 226200) -> 30000 点 (30.00 ms)
  m5_20: [201200, 231200) -> 30000 点 (30.00 ms)
  0_30: [206200, 236200) -> 30000 点 (30.00 ms)
  5_30: [211200, 241200) -> 30000 点 (30.00 ms)

开始计算特征...
  总耗时: 15.3 s
  worker 耗时: 15.3 s
  特征行数: 6
  特征列数: 640

单文件测试通过


## 7. 增量输出工具函数

In [6]:
def output_csv_for(label: str, part_idx: int) -> Path:
    # 根据标签和分片号定位当前批次的输出 CSV。
    cfg = next(item for item in DATASETS if item['label'] == label)
    return cfg['output_root'] / f'features_{label}_{RUN_TIMESTAMP}_part{part_idx:03d}.csv'


def append_dataframe_csv(path: Path, df: pd.DataFrame) -> None:
    # 追加写入 CSV；文件不存在时自动写表头。
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        mode='a',
        index=False,
        header=not path.exists(),
        encoding='utf-8-sig',
    )


def append_process_log(row: dict[str, object]) -> None:
    # 每个源文件处理结束后追加一行本地处理日志。
    append_dataframe_csv(PROCESS_LOG_CSV, pd.DataFrame([row]))


def cleanup_current_run_outputs() -> None:
    # 防止只重跑批处理单元时，对同一 RUN_TIMESTAMP 的 CSV 重复追加。
    if not OVERWRITE_CURRENT_RUN_OUTPUTS:
        return
    for cfg in DATASETS:
        for path in cfg['output_root'].glob(f'features_{cfg["label"]}_{RUN_TIMESTAMP}_part*.csv'):
            path.unlink()
    if PROCESS_LOG_CSV.exists():
        PROCESS_LOG_CSV.unlink()


def persist_one_file_result(rows: list[dict[str, object]], err: tuple[str, str, str] | None, elapsed_s: float) -> None:
    # 将单个源文件的处理结果立即落盘，并记录成功/失败状态。
    processed_at = datetime.now().isoformat(timespec='seconds')
    if err is not None:
        label, file_name, error = err
        append_process_log({
            'processed_at': processed_at,
            'run_timestamp': RUN_TIMESTAMP,
            'label': label,
            'source_file_name': file_name,
            'source_file_path': '',
            'output_part': '',
            'output_csv': '',
            'status': 'failed',
            'feature_rows': 0,
            'elapsed_s': round(elapsed_s, 3),
            'error': error,
        })
        return

    if not rows:
        append_process_log({
            'processed_at': processed_at,
            'run_timestamp': RUN_TIMESTAMP,
            'label': '',
            'source_file_name': '',
            'source_file_path': '',
            'output_part': '',
            'output_csv': '',
            'status': 'empty',
            'feature_rows': 0,
            'elapsed_s': round(elapsed_s, 3),
            'error': '',
        })
        return

    df_one = pd.DataFrame(rows)
    label = str(df_one['label'].iloc[0])
    part_idx = int(df_one['output_part'].iloc[0])
    output_csv = output_csv_for(label, part_idx)
    append_dataframe_csv(output_csv, df_one)

    append_process_log({
        'processed_at': processed_at,
        'run_timestamp': RUN_TIMESTAMP,
        'label': label,
        'source_file_name': str(df_one['source_file_name'].iloc[0]),
        'source_file_path': str(df_one['source_file_path'].iloc[0]),
        'output_part': part_idx,
        'output_csv': str(output_csv),
        'status': 'success',
        'feature_rows': len(df_one),
        'elapsed_s': round(elapsed_s, 3),
        'error': '',
    })

print('增量输出工具函数准备完成')


增量输出工具函数准备完成


## 8. 按文件持久化批处理

In [ ]:
max_workers = _auto_detect_workers() if FILE_WORKERS is None else int(FILE_WORKERS)
max_workers = max(1, max_workers)
cleanup_current_run_outputs()

print('=' * 60)
print('批处理开始：多标签单事件文件')
print(f'时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'标签数: {len(source_files_by_label)}')
print(f'源文件数: {len(source_files)}')
print(f'文件级并行进程数: {max_workers}')
print(f'单事件窗口 shared STFT/GPU: {ENABLE_SHARED_STFT}')
print(f'处理日志: {PROCESS_LOG_CSV}')
print('=' * 60)

all_rows = []
failed = []
output_files = set()
t_start = time.time()

if max_workers == 1:
    result_iter = (
        process_event_record_timed(
            record,
            BANDS,
            CUT_MODES,
            channel_index=CHANNEL_INDEX,
            enable_shared_stft=ENABLE_SHARED_STFT,
        )
        for record in source_records
    )
else:
    result_iter = Parallel(
        n_jobs=max_workers,
        prefer='processes',
        return_as='generator_unordered',
    )(
        delayed(process_event_record_timed)(
            record,
            BANDS,
            CUT_MODES,
            channel_index=CHANNEL_INDEX,
            enable_shared_stft=ENABLE_SHARED_STFT,
        )
        for record in source_records
    )

with tqdm(total=len(source_records), desc='处理并保存文件', unit='file') as pbar:
    for item in result_iter:
        rows, err, elapsed_s = item
        persist_one_file_result(rows, err, elapsed_s)
        all_rows.extend(rows)
        if err is not None:
            failed.append(err)
        elif rows:
            output_files.add(output_csv_for(str(rows[0]['label']), int(rows[0]['output_part'])))
        pbar.update(1)

t_elapsed = time.time() - t_start

df_all = pd.DataFrame(all_rows)
if len(df_all) > 0:
    df_all = df_all.sort_values(['label', 'source_file_index_in_label', 'window_mode']).reset_index(drop=True)

print(f'\n{"="*60}')
print('批处理完成')
print(f'总耗时: {t_elapsed:.1f} s ({t_elapsed/60:.1f} min)')
print(f'源文件数: {len(source_files)}')
print(f'特征行数: {len(df_all)}')
print(f'输出文件数: {len(output_files)}')
print(f'失败文件数: {len(failed)}')
print(f'处理日志: {PROCESS_LOG_CSV}')
if len(source_files) > 0:
    print(f'平均每个源文件耗时: {t_elapsed / len(source_files):.2f} s')
print(f'{"="*60}')

if failed:
    print('\n失败文件:')
    for label, name, err in failed:
        print(f'  {label} / {name}: {err}')


批处理开始：多标签单事件文件
时间: 2026-09-06 12:29:49
标签数: 4
源文件数: 195
文件级并行进程数: 14
单事件窗口 shared STFT/GPU: False
处理日志: e:\codes\ZZ-BK\outputs\DATA09_multi_label_single_event_features\_process_logs\process_log_20260906_122934.csv


处理并保存文件:   0%|          | 0/195 [00:00<?, ?file/s]

## 9. 输出汇总

In [ ]:
if PROCESS_LOG_CSV.exists():
    df_log = pd.read_csv(PROCESS_LOG_CSV)
    print(f'处理日志行数: {len(df_log)}')
    print(df_log['status'].value_counts().to_string())
else:
    df_log = pd.DataFrame()
    print('未找到处理日志')

if len(df_all) > 0:
    print(f'内存中特征行数: {len(df_all)}, 列数: {len(df_all.columns)}')
    print('\n按截取窗口统计行数:')
    print(df_all['window_mode'].value_counts().to_string())
    print('\n按标签统计源文件数:')
    print(df_all.groupby('label')['source_file_name'].nunique().to_string())
    print('\n按标签统计特征行数:')
    print(df_all['label'].value_counts().to_string())
else:
    print('内存中没有有效特征行')

print('\n输出 CSV 文件:')
for cfg in DATASETS:
    for path in sorted(cfg['output_root'].glob(f'features_{cfg["label"]}_{RUN_TIMESTAMP}_part*.csv')):
        print(f'  {path}')


## 10. 特征质量检查

In [ ]:
if len(df_all) > 0:
    meta_cols = {
        'source_file_name', 'source_file_path', 'label', 'source_file_index_in_label', 'output_part',
        'window_mode', 'window_start_ms', 'window_end_ms', 'window_n_samples',
        'sample_rate_hz', 'arrival_time', 'starttime', 'channel_index',
    }
    feature_cols = [c for c in df_all.columns if c not in meta_cols]
    print(f'特征列数: {len(feature_cols)}')

    full_nan = [c for c in feature_cols if pd.to_numeric(df_all[c], errors='coerce').isna().all()]
    if full_nan:
        print(f'警告: {len(full_nan)} 个特征全为 NaN')
        print(f'  {full_nan[:20]}')
    else:
        print('没有全 NaN 的特征列')

    nan_stats = []
    for col in feature_cols[:20]:
        series = pd.to_numeric(df_all[col], errors='coerce')
        nan_ratio = series.isna().sum() / len(series)
        nan_stats.append({'feature': col, 'nan_ratio': nan_ratio, 'mean': series.mean()})
    print('\n前 20 个特征统计:')
    print(pd.DataFrame(nan_stats).to_string(index=False))
else:
    print('没有有效特征行，跳过质量检查')


## 11. 架构与性能说明

```
多个标签目录中的 npz 文件 (npts, 2)
  -> 扫描 DATASETS 中配置的每个标签目录
  -> 按同一标签内的源 npz 文件数量分配 output_part
  -> 使用 joblib 进程 worker 并行处理源文件
  -> 每个源文件处理完成后，立即追加写入目标 CSV
  -> 同时向本地处理日志追加一行记录

围绕 arrival_time 截取三个窗口:
  [-10 ms, 20 ms]  (window_mode = m10_20)
  [  0 ms, 30 ms]  (window_mode = 0_30)
  [  5 ms, 35 ms]  (window_mode = 5_30)

存储:
  features_{label}_{timestamp}_partXXX.csv
  _process_logs/process_log_{timestamp}.csv
```

### 为什么使用按文件即时落盘

单事件特征提取在多标签、大量短文件场景下可能运行较久。每处理完一个源文件就写入 CSV，可以减少 notebook 中断造成的数据损失；本地处理日志也能直接追溯每个文件的完成时间、写入位置、耗时和错误信息。

### 为什么 CPU/GPU 利用率不同于 flow notebook

flow notebook 处理连续数据，单个文件能产生大量滑窗，因此 v2.2 流水线可以批量执行 STFT 并持续喂满 worker。当前 notebook 每个源文件只有三个短截取窗口，因此优化重点是跨源文件并行，默认走 CPU 进程并行。`ENABLE_SHARED_STFT` 可用于 GPU 路径实验，但短事件窗口通常难以让 GPU 长时间保持高利用率。
